# 움직임 시험 4 — 형태 조절(ControlNet canny) + 회색 실루엣: LoRA가 처음부터 그리고 형태만 블로킹을 따르게 (Kaggle·Colab T4, 학습 없음)
시험 1·2 결과(docs/DECISIONS-2026-10-26.md §6): 4색 블로킹은 '완성된 평면 그림'이라 낮은 강도에선 화풍이 안 입혀지고, IP-Adapter 참조는 색을 흔들었다.
- 조절 그림: 블로킹의 윤곽선(canny) — 불 덩어리 바깥선·밝기 단계 경계·빌딩 윤곽. 색은 넘기지 않는다.
- **A 처음부터**: 강도 1.0(블로킹 픽셀 안 씀) + 윤곽 조절 0.7 — LoRA가 색·화풍을 전부 그림.
- **B 회색 실루엣**: 블로킹을 회색 밝기 그림으로 바꿔 시작점으로(강도 0.85) + 윤곽 조절 0.6 — 밝기 구조는 남기고 색·화풍은 모델이 칠함.
- 같은 LoRA(3차 3바퀴)·글·시드(1234), 걸이 장은 같은 결과를 다시 씀.
- 설정(Kaggle): **GPU T4 x2**, **Internet On** → Save & Run All. 약 40–60분. Colab: 런타임 T4 GPU → 모두 실행(마지막 칸이 zip을 내려받게 함).
- 끝나면 `motion4_result.zip`(나란히 영상 `motion4.mp4` + 장별 그림 + 깜빡임 수치)을 Drive 공유 링크로 알려 준다.


In [ ]:
# 0) 어디서 돌리나: Kaggle이면 /kaggle/working, Colab이면 /content/out 에 결과를 쓴다
import os, glob
COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
OUT = "/content/out" if COLAB and not os.path.exists("/kaggle/working") else "/kaggle/working"
os.makedirs(OUT, exist_ok=True); print("결과 폴더", OUT)
# 1) 패키지와 자료: 기본 모델(HF), LoRA 3차 3바퀴·블로킹 두 가지(Drive)
# 버전 고정: Colab·Kaggle 기본 파이썬 3.13 + 새 transformers에서 diffusers 0.31이 깨졌다(FLAX_WEIGHTS_NAME, 2026-10-26). 서버 py3.13에서 아래 조합 import 확인
!pip install -q "diffusers==0.35.1" "transformers==4.56.2" "peft==0.17.1" "accelerate>=1.0" opencv-python-headless
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!curl -sSL -o /tmp/lora.safetensors "https://drive.usercontent.google.com/download?id=1ht_UqE8JkJTUX-eO5lNIqvhyFRcprUWJ&export=download&confirm=t"
import os, glob
for name, fid in [("num", "1lLCSZbiHE0QUO6xSw_uIeWiyUoNjlgDx"), ("clip", "1jITc3rXRduBnoW3C05I2XLmJ_7hEdtI6")]:
    os.system(f'curl -sSL -o /tmp/{name}.zip "https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t"')
    os.makedirs(f"/tmp/blk_{name}", exist_ok=True)
    os.system(f"cd /tmp/blk_{name} && unzip -q -o /tmp/{name}.zip '*.png'")
    print(name, len(glob.glob(f"/tmp/blk_{name}/*.png")), "장")
!ls -la /tmp/ilxl.safetensors /tmp/lora.safetensors


In [ ]:
# 2) 파이프라인: SDXL ControlNet 그림→그림 + LoRA 합치기 (윤곽 조절 xinsir/controlnet-canny-sdxl-1.0)
import torch
from diffusers import StableDiffusionXLControlNetImg2ImgPipeline, ControlNetModel, EulerAncestralDiscreteScheduler
from PIL import Image
cn = ControlNetModel.from_pretrained("xinsir/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_single_file("/tmp/ilxl.safetensors", controlnet=cn, torch_dtype=torch.float16)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights("/tmp/lora.safetensors", adapter_name="fx")
pipe.fuse_lora(lora_scale=1.0); pipe.unload_lora_weights()
pipe.to("cuda")
pipe.vae.to(torch.float32)
print(torch.cuda.memory_allocated() / 1e9, "GB")


In [ ]:
# 3) 두 방식으로 그리기
import numpy as np, json, cv2
P = "nkmr_fx, vivid colors, anime screencap, explosion, fire, embers, white core orange rim, layered cel shading, no outline, sharp tips, large effect, no humans, city, night, wide shot"
N = "lowres, worst quality, text, watermark"
STEPS, CFG, SEED = 28, 6.0, 1234
def edges(b):  # 블로킹 윤곽선(색 없음)
    g = cv2.cvtColor(np.asarray(b), cv2.COLOR_RGB2GRAY)
    e = cv2.Canny(g, 30, 90)
    return Image.fromarray(np.stack([e] * 3, -1))
def grey(b):  # 회색 실루엣: 밝기만 남기고 대비를 줄여 모델이 색을 칠할 여지를 둠
    g = cv2.cvtColor(np.asarray(b), cv2.COLOR_RGB2GRAY).astype(np.float32)
    g = 40 + g * 0.7
    return Image.fromarray(np.stack([g.clip(0, 255).astype(np.uint8)] * 3, -1))
def run(init, ctrl, s, cs):
    g = torch.Generator("cuda").manual_seed(SEED)
    with torch.inference_mode():
        return pipe(prompt=P, negative_prompt=N, image=init, control_image=ctrl, strength=s, controlnet_conditioning_scale=cs,
                    num_inference_steps=STEPS, guidance_scale=CFG, generator=g, width=1344, height=768).images[0]
def diff(a, b):
    return float(np.abs(np.asarray(a, np.float32) - np.asarray(b, np.float32)).mean())
log = {}
for name in ("num", "clip"):
    blk = [Image.open(p).convert("RGB") for p in sorted(glob.glob(f"/tmp/blk_{name}/*.png"))]
    for tag, mk in (("A_scratch", lambda b: (b, edges(b), 1.0, 0.7)), ("B_grey", lambda b: (grey(b), edges(b), 0.85, 0.6))):
        cache, ims = {}, []
        for b in blk:
            key = np.asarray(b).tobytes()
            if key not in cache:
                cache[key] = run(*mk(b))
            ims.append(cache[key])
        d = f"{OUT}/m4/{name}/{tag}"; os.makedirs(d, exist_ok=True)
        for i, im in enumerate(ims):
            im.save(f"{d}/f{i:03d}.png")
        fl = np.mean([abs(diff(ims[i + 1], ims[i]) - diff(blk[i + 1], blk[i])) for i in range(len(ims) - 1)])
        log[f"{name} {tag}"] = round(float(fl), 2)
        print(name, tag, len(ims), "장, 깜빡임", log[f"{name} {tag}"])
    for sub, f in (("blocking", lambda b: b), ("edges", edges), ("grey", grey)):
        d0 = f"{OUT}/m4/{name}/{sub}"; os.makedirs(d0, exist_ok=True)
        for i, b in enumerate(blk):
            f(b).save(f"{d0}/f{i:03d}.png")
json.dump(log, open(f"{OUT}/motion4_flicker.json", "w"), ensure_ascii=False, indent=1)


In [ ]:
# 4) 나란히 영상: 위 줄 num, 아래 줄 clip / 칸 = 블로킹 | 윤곽(조절 그림) | A 처음부터 | B 회색 실루엣. 24fps, 3번 반복
rows = {n: {k: sorted(glob.glob(f"{OUT}/m4/{n}/{k}/*.png")) for k in ("blocking", "edges", "A_scratch", "B_grey")} for n in ("num", "clip")}
def rd(p, label):
    im = cv2.resize(cv2.imread(p), (336, 192)); cv2.putText(im, label, (6, 20), 0, 0.55, (255, 255, 255), 2); return im
T = max(len(rows[n]["blocking"]) for n in rows)
vw = cv2.VideoWriter(f"{OUT}/motion4.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 24, (1344, 384))
for _ in range(3):
    for t in range(T):
        vw.write(np.vstack([np.hstack([rd(v[t % len(v)], f"{n} {k}") for k, v in rows[n].items()]) for n in rows]))
vw.release()
!cd {OUT} && zip -q -r motion4_result.zip m4 motion4.mp4 motion4_flicker.json && ls -la {OUT}


In [ ]:
# 5) Colab이면 결과 zip 내려받기(Kaggle은 Output 탭에서 받음)
if COLAB and OUT.startswith("/content"):
    from google.colab import files
    files.download(f"{OUT}/motion4_result.zip")
